In [13]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models
import gradio as gr

In [14]:
def build_model(num_classes=15, Weight_name='',freeze_layers=-30):
    pretrained_model = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights=None,
        input_shape=(300, 300, 3),
    )

    pretrained_model.trainable = True

    # Freeze layer awal
    for layer in pretrained_model.layers[:freeze_layers]:
        layer.trainable = False

    model = models.Sequential([
        pretrained_model,
        layers.GlobalAveragePooling2D(),

        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation="softmax")
    ])
    model.load_weights(Weight_name)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    print('LOAD SUCESSFULL!!')
    return model

def gradcam_heatmap(img_array, model, base_model, last_conv_name):
    last_conv_layer = base_model.get_layer(last_conv_name)

    x = base_model.output
    head_layers = model.layers[1:]

    for layer in head_layers:
        x = layer(x)

    grad_model = tf.keras.models.Model(
        inputs=base_model.input,
        outputs=[last_conv_layer.output, x]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        pred_output = predictions[:, pred_index]

    # Grads shape = (1, H, W, channels)
    grads = tape.gradient(pred_output, conv_outputs)

    # Compute channel importance
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]  # Drop batch dim

    # Weighted sum of channels
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # Normalize
    heatmap = np.maximum(heatmap.numpy(), 0)
    denom = heatmap.max() if heatmap.max() != 0 else 1e-10
    heatmap /= denom

    return heatmap

def get_heatmap_img(model, image_path):
    image = tf.keras.preprocessing.image.load_img(image_path)
    # Convert PIL to numpy
    if not isinstance(image, np.ndarray):
        image = np.array(image)

    # Resize to model input size
    image_resized = tf.image.resize(image, (300, 300)).numpy()

    # EfficientNet preprocessing
    img_array = tf.keras.applications.efficientnet.preprocess_input(image_resized)
    img_array = np.expand_dims(img_array, axis=0)

    base_model = model.layers[0]    # EfficientNetB3
    last_conv_name = "top_conv"

    heatmap = gradcam_heatmap(
        img_array,
        model=model,
        base_model=base_model,
        last_conv_name=last_conv_name
    )

    # Resize heatmap ke ukuran asli
    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))

    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)

    # Convert image to uint8 (0-255)
    base_img = image
    if base_img.max() <= 1:
        base_img = (base_img * 255).astype(np.uint8)

    # Overlay
    superimposed_img = (heatmap_color * 0.4 + base_img).astype(np.uint8)

    return superimposed_img

def get_label(pred):
    label = ['Africanized Honey Bees (Killer Bees)', 'Aphids', 'Armyworms', 'Brown Marmorated Stink Bugs', 'Cabbage Loopers', 'Citrus Canker', 'Colorado Potato Beetles', 'Corn Borers', 'Corn Earworms', 'Fall Armyworms','Fruit Flies','Spider Mites','Thrips','Tomato Hornworms','Western Corn Rootworms']
    return label[pred]

def get_description(label):
    desc = {
        "Africanized Honey Bees (Killer Bees)" : "Variasi agresif dari lebah madu (Apis mellifera scutellata). Banyak ditemukan di Amerika. Sangat defensif terhadap gangguan.",
        'Aphids' : 'Serangga kecil penghisap getah yang berkembang biak sangat cepat. Menyerang sayuran, buah, dan tanaman hias.',
        'Armyworms' : 'Larva dari beberapa spesies ngengat. Memakan daun tanaman secara berkelompok dan dapat menyebabkan kerusakan besar.',
        'Brown Marmorated Stink Bugs': 'Hama invasif yang menyerang buah dan sayuran. Menghasilkan bau tidak enak saat terganggu.',
        'Cabbage Loopers' : 'Larva ngengat yang menyerang kubis dan sayuran daun. Memiliki gerakan seperti “looping”.',
        'Citrus Canker' : "Penyakit bakteri (Xanthomonas axonopodis) yang menyerang daun, batang, dan buah jeruk.",
        'Colorado Potato Beetles' : 'Hama utama tanaman kentang. Sangat rakus dan resisten terhadap banyak insektisida.',
        'Corn Borers' : 'Larva yang masuk ke batang jagung dan melemahkan struktur tanaman.',
        'Corn Earworms' : 'Larva ngengat yang menyerang tongkol jagung, sering muncul pada ujung tongkol.',
        'Fall Armyworms' : 'Spesies invasif berbahaya (Spodoptera frugiperda). Menyerang banyak jenis tanaman, terutama jagung.',
        'Fruit Flies' : 'Menyerang buah matang dengan meletakkan telur di dalamnya; larva memakan daging buah.',
        'Spider Mites' : 'Tungau mikro penghisap cairan sel daun; berkembang cepat saat cuaca kering.',
        'Thrips' : 'Serangga kecil yang mengikis permukaan daun & bunga serta pembawa virus TSWV atau Tomato Spotted Wilt Virus (Virus bercak layu tomat).',
        'Tomato Hornworms' : 'Ulat besar yang memakan daun dan buah tomat; kamuflase sangat baik.',
        'Western Corn Rootworms' : 'Larva merusak akar jagung, menyebabkan tanaman tidak stabil.'
    }
    return desc[label]

def get_handleStrategy(label):
    desc = {
        "Africanized Honey Bees (Killer Bees)" : ["Hindari mengganggu sarang.", "Gunakan pakaian pelindung.", "Panggil ahli pengendali hama untuk pemindahan sarang.", "Pasang perangkap feromon jika diperlukan."],
        'Aphids' : ['Introduksi predator (ladybugs).', 'Sabun insektisida.', 'Neem oil.', 'Tanaman perangkap.', 'Buang daun terinfeksi berat.'],
        'Armyworms' : ['Monitoring malam hari.', 'Aplikasi Bacillus thuringiensis (Bt).', 'Insektisida berbasis spinosad, rotasi tanaman.'],
        'Brown Marmorated Stink Bugs': ['Perangkap feromon, jaring tanaman.', 'Insektisida pyrethroid.', 'Eliminasi tempat persembunyian.'],
        'Cabbage Loopers' : ['Aplikasi Bacillus thuringiensis (Bt).', 'Neem oil', 'Monitoring telur di bawah daun.', 'Pemusnahan manual.'],
        'Citrus Canker' : ["Pemangkasan bagian terinfeksi.", 'Semprot tembaga, sanitasi kebun.', "Kontrol penyebaran melalui alat & angin."],
        'Colorado Potato Beetles' : ['Rotasi insektisida', 'Mulsa reflektif', 'Pemetikan larva/telur manual.', 'Tanaman perangkap.'],
        'Corn Borers' : ['Jagung Bt', 'Rotasi tanaman.', 'Pemangkasan residu panen.', 'Monitoring feromon.'],
        'Corn Earworms' : ['Aplikasi Bacillus thuringiensis (Bt).', 'Penggunaan predator alami.', 'Perlindungan ujung tongkol dengan minyak mineral.'],
        'Fall Armyworms' : ['Monitoring feromon.', 'Insektisida sistemik.', 'Varietas tahan.', 'Manajemen lahan rutin.'],
        'Fruit Flies' : ['Perangkap umpan (cuka/ragi).', 'Sanitasi area.', 'Jaring buah.', 'Protein bait + insektisida selektif.'],
        'Spider Mites' : ['Tingkatkan kelembapan.', 'Neem oil.', 'Sabun insektisida.', 'Predator Phytoseiulus persimilis.'],
        'Thrips' : ['Sticky trap biru', 'Spinosad', 'Neem oil', 'Predator Orius', 'Sanitasi bunga rusak.'],
        'Tomato Hornworms' : ['Ambil manual', 'Aplikasi Bacillus thuringiensis (Bt)', 'Biarkan parasitoid (casing putih di tubuh ulat).', 'Tanaman penolak seperti basil.'],
        'Western Corn Rootworms' : ['Rotasi tanaman.', 'Jagung Bt', 'Insektisida granular', 'Manajemen residu panen.']
    }
    return desc[label]

def get_ecoLoss(label):
    desc = {
        "Africanized Honey Bees (Killer Bees)" : ["Risiko keselamatan tinggi bagi pekerja kebun.", "Dapat mengganggu aktivitas pertanian."],
        'Aphids' : ['Penurunan hasil panen.', 'Penyebaran virus tanaman.', 'Kualitas buah menurun.'],
        'Armyworms' : ['Kerusakan cepat pada lahan luas.', 'Terutama jagung dan sayuran.'],
        'Brown Marmorated Stink Bugs': ['Penurunan kualitas buah.', 'Kerugian besar pada apel, peach, tomat.'],
        'Cabbage Loopers' : ['Berkurangnya ukuran dan kualitas kubis', 'Potensi kegagalan panen.'],
        'Citrus Canker' : ["Kehilangan hasil tinggi.", "Hambatan ekspor.", "Penyebaran cepat di perkebunan besar."],
        'Colorado Potato Beetles' : ['Sangat resisten insektisida', 'Potensi gagal panen kentang.'],
        'Corn Borers' : ['Penurunan hasil.', 'Penurunan kualitas tongkol.', 'Infeksi sekunder oleh jamur.'],
        'Corn Earworms' : ['Menurunkan kualitas jagung konsumsi.', 'Terutama untuk pasar segar.'],
        'Fall Armyworms' : ['Kerugian terbesar pada jagung.', 'Penyebaran cepat antar area.'],
        'Fruit Flies' : ['Kerugian besar di buah tropis (mangga, jambu, pepaya).'],
        'Spider Mites' : ['Kerusakan intens pada tomat, melon, stroberi.'],
        'Thrips' : ['Menurunkan kualitas bunga', 'Potensi gagal panen karena virus.'],
        'Tomato Hornworms' : ['Kerusakan parah dalam waktu singkat pada tomat dan paprika.'],
        'Western Corn Rootworms' : ['Salah satu hama jagung paling merugikan di dunia.']
    }
    return desc[label]

def get_dangerLev(label):
    desc = {
        "Africanized Honey Bees (Killer Bees)": "Extreme",
        "Aphids": "Medium",
        "Armyworms": "High",
        "Brown Marmorated Stink Bugs": "Medium",
        "Cabbage Loopers": "Medium",
        "Citrus Canker": "High",
        "Colorado Potato Beetles": "High",
        "Corn Borers": "High",
        "Corn Earworms": "High",
        "Fall Armyworms": "Extreme",
        "Fruit Flies": "High",
        "Spider Mites": "High",
        "Thrips": "Medium",
        "Tomato Hornworms": "Medium",
        "Western Corn Rootworms": "High"
    }
    return desc[label]
def list_to_bullets(items):
    return "\n".join([f"- {i}" for i in items])

In [ ]:
weight_name = "efficientnetb3_model_weights.h5"
loaded_model = build_model(
    num_classes=15,*
    Weight_name=weight_name,
    freeze_layers=-30
)
print("Model loaded OK.")

def predict(image_path,state,model=loaded_model):
    if image_path is None:
        print("TIDAK ADA GAMBAR")
        return None, "Tidak ada gambar", "", "", ""

    image = tf.keras.preprocessing.image.load_img(image_path)
    image_array= tf.keras.preprocessing.image.img_to_array(image)
    img_resized = tf.image.resize(image_array, (300, 300)).numpy()
    img_pp = preprocess_input(img_resized)
    pred = model.predict(np.expand_dims(img_pp, axis=0))[0]

    idx = np.argmax(pred)
    label = get_label(idx)

    # buat heatmap
    overlay = get_heatmap_img(model, image_path)
    dampak = list_to_bullets(get_ecoLoss(label))
    penanganan = list_to_bullets(get_handleStrategy(label))
    # simpan history
    result = (
        f"Nama : {label}\n"
        f"Deskripsi: {get_description(label)}\n"
        f"Tingkat Bahaya: {get_dangerLev(label)}\n"
        f"Dampak: \n{dampak}\n"
        f"Cara menangani: \n{penanganan}"
    )
    lines = result.strip().split("\n")
    summary = "\n".join(lines[:2]) if len(lines) >= 2 else result
    if image:
        state.append((image,summary))
    return overlay, result ,state, state  


with gr.Blocks(title="Pest Detector") as demo:

    gr.Markdown("# 🐛 FarmPest Guide")
    gr.Markdown("""
                ## Selamat Datang di Aplikasi FarmPest Guide  
                Aplikasi ini merupakan sistem deteksi hama dan penyakit tanaman berbasis AI. Pengguna dapat mengupload gambar tanaman yang terdampak, dan sistem akan bisa mengidentifikasi jenis hama atau penyakit, menampilkan tingkat bahayanya, serta memberikan deskripsi singkat, potensi kerugian, dan rekomendasi penanganan. Aplikasi ini dirancang untuk membantu petani, agronomis, dan praktisi lapangan dalam mengambil keputusan cepat dan akurat untuk menjaga kesehatan tanaman dan meningkatkan hasil panen.

                Anggota Kelompok:
                - David Goanli - 2702223582
                - Kelson - 2702245135
                - Rafael Komala - 2702227611
                """)
    with gr.Tabs():

        # ========== TAB 1: Prediksi ==========
        with gr.Tab("Prediksi"):
            with gr.Row():

                # ---- INPUT ----
                with gr.Column():
                    input_path = gr.Image(label="Upload Gambar", type="filepath")
                    predict_btn = gr.Button("Prediksi", variant="primary")
                    reset_btn = gr.ClearButton([input_path])
                    gr.Examples(
                        examples=[
                            ["example/Image_1.jpg"],
                            ["example/Image_2.jpg"],
                            ["example/Image_3.jpg"],
                        ],
                        inputs=[input_path],
                        label="Contoh Gambar"
                    )
                # ---- OUTPUT ----
                with gr.Column():
                    output_img = gr.Image(label="GradCAM Overlay")
                    output = gr.Textbox(label="Hasil Analisis Hama", lines=12, interactive=False)

            
            state = gr.State([])

        # ========== TAB 2: Riwayat ==========
        with gr.Tab("Riwayat Prediksi"):
            history = gr.Gallery(label="Riwayat Prediksi", columns=2, object_fit="cover")
    
    predict_btn.click(
                fn=predict,
                inputs=[input_path,state],
                outputs=[output_img, output,state, history]
            )
    # Footer
    gr.Markdown("---")
    gr.Markdown("""
    ## **Model Info:** EfficientNetB3 + Custom Classifier  
    ## **Cara Pakai:** 
    ## 1. Upload gambar pada kolom yang telah disediakan 
    ## 2. Tekan tombol Prediksi untuk memprediksi gambar yang telah dimasukan
    ## 3. Tunggu sebentar dan hasilnya akan keluar pada kolom sebelah kanan 
    """)

demo.launch()

LOAD SUCESSFULL!!
Model loaded OK.
* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
